## Exploratory Data Analysis

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet("consolidated_logistics_base.parquet")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe(include="all").T

In [ ]:
#Getting an idea of how much are the late orders.
df["is_late"].value_counts(dropna=False, normalize=True) 

In [ ]:
#Analysing Nulls
nulls = df.isna().mean().sort_values(ascending=False)
nulls[nulls > 0].head(30)

In [ ]:
# Plot null ratios
plt.figure(figsize=(8, 10))
sns.barplot(x=nulls[nulls > 0], y=nulls[nulls > 0].index)
plt.title("Missing values ratio by column")
plt.xlabel("Missing ratio")
plt.ylabel("Column")
plt.tight_layout()
plt.show()

### Missing values - main observations:

- Some columns have a non‑negligible proportion of missing values, especially date fields related to delivery and review text fields.
- Columns with missing values are mostly expected (e.g. orders not delivered or without reviews), so they are likely informative rather than random noise.
- There are no columns with an extremely high null ratio that would be immediate candidates for dropping at this stage.
- We will need to handle missing values carefully during feature engineering, possibly with separate indicators (flags) for “missing” where it makes business sense.

In [ ]:
# Derive delivery time columns to enhance features

df["actual_delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["estimated_delivery_days"] = (
    df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
).dt.days

df["delay_vs_estimated_days"] = (
    df["actual_delivery_days"] - df["estimated_delivery_days"]
)

In [ ]:
# Descriptive statistics for delivery times
df[["actual_delivery_days", "estimated_delivery_days", "delay_vs_estimated_days"]].describe()

In [ ]:
# Boxplot of delay vs estimated delivery days
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df[df["delay_vs_estimated_days"].notna()],
    y="delay_vs_estimated_days"
)
plt.axhline(0, color="red", linestyle="--", label="On time threshold")
plt.title("Delay vs Estimated Delivery (days)")
plt.ylabel("Days (positive = late, negative = early)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot delay distribution split by is_late
plt.figure(figsize=(10, 4))
sns.histplot(
    data=df[df["is_late"].notna()],
    x="delay_vs_estimated_days",
    hue="is_late",
    bins=40,
    kde=False,
    multiple="layer"
)
plt.xlim(-10, 20)
plt.title("Delay vs estimated delivery, split by is_late")
plt.xlabel("Days (actual - estimated)")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate late rate by purchase month
df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M")

monthly = (
    df[df["is_late"].notna()]
    .groupby("purchase_month")
    .agg(
        total_orders=("order_id", "nunique"),
        late_orders=("is_late", lambda x: (x == 1).sum()),
        ontime_orders=("is_late", lambda x: (x == 0).sum())
    )
)

monthly["late_rate"] = monthly["late_orders"] / (
    monthly["late_orders"] + monthly["ontime_orders"]
)
monthly.index = monthly.index.to_timestamp()

# Plot monthly late rate
plt.figure(figsize=(10, 4))
sns.lineplot(data=monthly, x=monthly.index, y="late_rate", marker="o")
plt.title("Late delivery rate over time")
plt.xlabel("Purchase month")
plt.ylabel("Late rate")
plt.tight_layout()
plt.show()

### Time series analysis – main findings:

- The `delay_vs_estimated_days` variable shows that many orders are delivered slightly before or very close to the estimated date, but there is a noticeable right tail of late deliveries.
- The histogram of `delay_vs_estimated_days` indicates that small positive delays (a few days) are relatively common, while very large delays are less frequent but still present.
- The boxplot split by `is_late` shows a clear separation between on‑time and late orders: late orders have a much higher median delay and a wider spread, which validates `delay_vs_estimated_days` as a strong candidate feature for the model.
- There is a peak of delays in an specific period between the end of 2017 and the firs trimester of 2018.

In [ ]:
# Aggregate late rate by customer state
state_late = (
    df[df["is_late"].notna()]
    .groupby("customer_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_orders=("is_late", lambda x: (x == 1).sum())
    )
)

state_late["late_rate"] = state_late["late_orders"] / state_late["total_orders"]
state_late = state_late.sort_values("late_rate", ascending=False)

state_late

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# --- Customer state ---
sns.barplot(
    data=state_late.reset_index(),
    x="customer_state",
    y="late_rate",
    order=state_late.index,
    ax=axes[0]
)

# Percentual annotations
for p in axes[0].patches:
    axes[0].annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=7, rotation=90
    )

axes[0].set_title("Late delivery rate by customer state")
axes[0].set_xlabel("Customer state")
axes[0].set_ylabel("Late rate")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

# --- Seller state ---
seller_state_late = (
    df[df["is_late"].notna()]
    .groupby("seller_geo_state")
    .agg(
        total_orders=("order_id", "nunique"),
        late_orders=("is_late", lambda x: (x == 1).sum())
    )
)
seller_state_late["late_rate"] = (
    seller_state_late["late_orders"] / seller_state_late["total_orders"]
)
seller_state_late = seller_state_late.sort_values("late_rate", ascending=False)

sns.barplot(
    data=seller_state_late.reset_index(),
    x="seller_geo_state",
    y="late_rate",
    order=seller_state_late.index,
    ax=axes[1]
)

# Percentual annotations
for p in axes[1].patches:
    axes[1].annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=7, rotation=90
    )

axes[1].set_title("Late delivery rate by seller state")
axes[1].set_xlabel("Seller state")
axes[1].set_ylabel("Late rate")
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

plt.suptitle("Late rate by geography", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df["seller_customer_distance_km"].describe()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(
    data=df,
    x="seller_customer_distance_km",
    bins=100,
    kde=True
)
plt.title("Distribution of seller–customer distance (km)")
plt.xlabel("Distance (km)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(
    data=df,
    x="is_late",
    y="seller_customer_distance_km"
)
plt.title("Seller–customer distance vs late status")
plt.xlabel("Is late (0 = on time, 1 = late)")
plt.ylabel("Distance (km)")
plt.tight_layout()
plt.show()

In [ ]:
# Filter out NA before bucketing
df_geo = df[df["is_late"].notna() & df["seller_customer_distance_km"].notna()].copy()

# Late rate by distance bucket
bins   = [0, 50, 200, 500, 1000, 2000, 5000, 10000]
labels = ["0–50km", "50–200km", "200–500km", "500km–1k", "1k–2k", "2k–5k", "5k+"]

df_geo["distance_bucket_km"] = pd.cut(
    df_geo["seller_customer_distance_km"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

distance_late = (
    df_geo
    .groupby("distance_bucket_km", observed=True)
    .agg(
        total_orders=("is_late", "count"),
        late_rate=("is_late", lambda x: (x == 1).sum() / len(x))  # evita ambiguidade do NA
    )
    .reset_index()
)

print(distance_late)

# Plot
fig, ax1 = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=distance_late,
    x="distance_bucket_km",
    y="late_rate",
    color="steelblue",
    ax=ax1
)

for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=9
    )

ax1.set_title("Late rate by seller–customer distance bucket")
ax1.set_xlabel("Distance bucket (km)")
ax1.set_ylabel("Late rate")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

ax2 = ax1.twinx()
ax2.plot(
    distance_late.index,
    distance_late["total_orders"],
    color="orange",
    marker="o",
    linewidth=2,
    label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

### Geolocation analysis – main findings:

- **Distance matters**: Although the boxplot indicates similar behavior in within late or not orders, the barplot shows that the distance between seller and customer has a clear relationship with late deliveries; larger distance buckets exhibit significantly higher late rates compared to local/regional deliveries.
- **Seller hotspot of delays**: the seller state with the highest late rate, by a wide margin, is **Amazonas (AM)**, suggesting relevant logistical challenges for shipments originating from this region.
- **Customer hotspots of delays**: customer states in the **Northeast** concentrate some of the highest late rates in the dataset, indicating potential bottlenecks in the delivery network to that region.
- These findings reinforce the importance of including **geography and seller–customer distance** as core features in the delay prediction model.

In [ ]:
df.columns

In [ ]:
cols_size = [
    "product_weight_g",
    "product_volume_cm3",
    "is_heavy_product",
    "is_bulky_product",
    "is_late",
]

df_eda = df[cols_size].dropna(subset=["is_late"]).copy()
df_eda["is_late"] = df_eda["is_late"].astype(int)
df_eda.head()

In [ ]:
df_eda.groupby("is_late")[
    ["product_weight_g", "product_volume_cm3"]
].agg(["mean", "median", "std", "min", "max"]).T

In [ ]:
plt.figure(figsize=(5, 4))
sns.boxplot(
    data=df_eda,
    x="is_late",
    y="product_weight_g",
)
plt.xlabel("Is late (0 = on time, 1 = late)")
plt.ylabel("product_weight_g")
plt.title("Product weight vs late status")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, feat in zip(axes, ["product_weight_g", "product_volume_cm3"]):
    sns.boxplot(data=df_eda, x="is_late", y=feat, ax=ax)
    ax.set_xlabel("Is late (0 = on time, 1 = late)")
    ax.set_ylabel(feat)
    ax.set_title(f"{feat} vs is_late")

plt.suptitle("Product weight and volume vs late status", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
bins = [0, 1e3, 5e3, 1e4, 5e4, 1e5, df_eda["product_volume_cm3"].max()]
labels = ["0–1k", "1k–5k", "5k–10k", "10k–50k", "50k–100k", "100k+"]

df_eda["product_volume_bucket"] = pd.cut(
    df_eda["product_volume_cm3"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

vol_late = (
    df_eda
    .groupby("product_volume_bucket", observed=True)
    .agg(
        total_orders=("is_late", "count"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

vol_late

In [ ]:
# Late rate by product volume bucket (more granular)
bins   = [0, 500, 1e3, 2e3, 5e3, 1e4, 2e4, 5e4, 1e5, df_eda["product_volume_cm3"].max()]
labels = ["0–500", "500–1k", "1k–2k", "2k–5k", "5k–10k", "10k–20k", "20k–50k", "50k–100k", "100k+"]

df_eda["product_volume_bucket"] = pd.cut(
    df_eda["product_volume_cm3"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

vol_late = (
    df_eda
    .groupby("product_volume_bucket", observed=True)
    .agg(
        total_orders=("is_late", "count"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

print(vol_late)

# Plot
fig, ax1 = plt.subplots(figsize=(12, 5))

sns.barplot(
    data=vol_late,
    x="product_volume_bucket",
    y="late_rate",
    color="steelblue",
    ax=ax1,
)

for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=8
    )

ax1.set_xlabel("Product volume bucket (cm³)")
ax1.set_ylabel("Late rate")
ax1.set_title("Late rate by product volume bucket (granular)")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax1.tick_params(axis="x", rotation=30)

ax2 = ax1.twinx()
ax2.plot(
    vol_late.index,
    vol_late["total_orders"],
    color="orange",
    marker="o",
    linewidth=2,
    label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Late rate by product volume bucket (2 buckets: <=50k and >50k cm³)
bins   = [0, 5e4, df_eda["product_volume_cm3"].max()]
labels = ["0–50k", "50k+"]

df_eda["product_volume_bucket"] = pd.cut(
    df_eda["product_volume_cm3"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

vol_late_2 = (
    df_eda
    .groupby("product_volume_bucket", observed=True)
    .agg(
        total_orders=("is_late", "count"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

print(vol_late_2)

# Bar chart: late rate + volume line
fig, ax1 = plt.subplots(figsize=(6, 4))

sns.barplot(
    data=vol_late_2,
    x="product_volume_bucket",
    y="late_rate",
    color="steelblue",
    ax=ax1,
)

for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=9
    )

ax1.set_xlabel("Product volume bucket (cm³)")
ax1.set_ylabel("Late rate")
ax1.set_title("Late rate by product volume bucket (<=50k vs >50k)")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

ax2 = ax1.twinx()
ax2.plot(
    vol_late_2.index,
    vol_late_2["total_orders"],
    color="orange",
    marker="o",
    linewidth=2,
    label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# Late rate by product weight bucket (2 buckets: <=10k and >10k g)
bins   = [0, 1e4, df_eda["product_weight_g"].max()]
labels = ["0–10k", "10k+"]

df_eda["product_weight_bucket"] = pd.cut(
    df_eda["product_weight_g"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

weight_late = (
    df_eda
    .groupby("product_weight_bucket", observed=True)
    .agg(
        total_orders=("is_late", "count"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

print(weight_late)

# Bar chart: late rate + volume line
fig, ax1 = plt.subplots(figsize=(6, 4))

sns.barplot(
    data=weight_late,
    x="product_weight_bucket",
    y="late_rate",
    color="steelblue",
    ax=ax1,
)

for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=9
    )

ax1.set_xlabel("Product weight bucket (g)")
ax1.set_ylabel("Late rate")
ax1.set_title("Late rate by product weight (<=10k vs >10k g)")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

ax2 = ax1.twinx()
ax2.plot(
    weight_late.index,
    weight_late["total_orders"],
    color="orange",
    marker="o",
    linewidth=2,
    label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Late rate by product category
category_late = (
    df[df["is_late"].notna()]
    .groupby("product_category_name")
    .agg(
        total_orders=("order_id", "nunique"),
        late_rate=("is_late", "mean"),
    )
    .sort_values("late_rate", ascending=False)
    .reset_index()
)

print(category_late.head(15))

# Plot top 20 categories by late rate
fig, ax1 = plt.subplots(figsize=(12, 6))

top20 = category_late.head(20)

sns.barplot(
    data=top20,
    x="product_category_name",
    y="late_rate",
    color="steelblue",
    ax=ax1,
)

for p in ax1.patches:
    ax1.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=7
    )

ax1.set_xlabel("Product category")
ax1.set_ylabel("Late rate")
ax1.set_title("Late rate by product category (top 20)")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))
ax1.tick_params(axis="x", rotation=90)

ax2 = ax1.twinx()
ax2.plot(
    top20.index,
    top20["total_orders"],
    color="orange",
    marker="o",
    linewidth=2,
    label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

### Product analysis – main findings

- **Volume and late rate**: the relationship between product volume and late delivery
  rate is not strictly monotonic. Mid-range volumes show comparable late rates to
  larger ones, suggesting volume alone is not a strong linear predictor.

- **Weight threshold**: orders with products weighing more than 10kg show a
  noticeably higher late rate, consistent with the hypothesis that heavier shipments
  face additional logistics constraints (e.g. special handling, carrier restrictions).

- **Feature engineering decision**: both continuous features (`product_weight_g`,
  `product_volume_cm3`) and binary flags (`is_heavy_product`, `is_bulky_product`)
  were retained. This allows linear models to use the thresholds directly while
  tree-based models can explore non-linear interactions.

- **Category late rate**: categories like `moveis_colchao_e_estofado` and
  `fashion_underwear_e_moda_praia` show consistently high late rates with
  sufficient volume to be statistically reliable. Low-volume categories such as
  `casa_conforto_2` should be interpreted with caution as their rates may reflect
  sampling noise rather than a structural pattern.

- **Similar category names**: some categories share near-identical names with a
  numeric suffix (e.g. `casa_conforto` and `casa_conforto_2`). These were
  intentionally kept separate, as the suffix may indicate distinct product lines
  with different logistics profiles. 

- **Feature engineering decision (category)**: given the high cardinality of
  `product_category_name`, target encoding (`category_late_rate`) is preferred
  over one-hot encoding to avoid dimensionality explosion while preserving
  predictive signal.

In [ ]:
df.columns

In [ ]:
df['freight_value'].describe()

In [ ]:
# Late rate by payment type
payment_late = (
    df[df["is_late"].notna()]
    .groupby("payment_type_main")
    .agg(
        total_orders=("order_id", "nunique"),
        late_rate=("is_late", "mean"),
    )
    .sort_values("late_rate", ascending=False)
    .reset_index()
)

print(payment_late)

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=payment_late,
    x="payment_type_main",
    y="late_rate",
    color="steelblue",
    ax=ax,
)

for p in ax.patches:
    ax.annotate(
        f"{p.get_height():.1%}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=9
    )

ax.set_xlabel("Payment type")
ax.set_ylabel("Late rate")
ax.set_title("Late rate by payment type")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

In [ ]:
# Late rate by price bin
df["price_bin"] = pd.qcut(df["price"], q=10, duplicates="drop")

price_late = (
    df[df["is_late"].notna()]
    .groupby("price_bin", observed=True)
    .agg(
        total_orders=("order_id", "nunique"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

print(price_late)

fig, ax1 = plt.subplots(figsize=(12, 5))

sns.barplot(
    data=price_late,
    x=price_late.index,
    y="late_rate",
    color="steelblue",
    ax=ax1,
)

ax1.set_xticks(price_late.index)
ax1.set_xticklabels(
    [str(b) for b in price_late["price_bin"]],
    rotation=45, ha="right", fontsize=7
)
ax1.set_ylabel("Late rate")
ax1.set_xlabel("Price bin (deciles)")
ax1.set_title("Late rate by price decile")
ax1.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

ax2 = ax1.twinx()
ax2.plot(
    price_late.index,
    price_late["total_orders"],
    color="orange", marker="o", linewidth=2, label="Order volume"
)
ax2.set_ylabel("Order volume", color="orange")
ax2.tick_params(axis="y", labelcolor="orange")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

### Payment type & price findings

- **Payment type**: late rates range narrowly from 7.2% (voucher) to 8.6% (boleto),
  suggesting payment type carries limited standalone predictive signal. The slight
  boleto premium may reflect processing delays inherent to this payment method.
  Feature will be retained as a low-cost categorical input.

- **Price**: late rate shows a mild positive linear trend with price, rising from
  ~7.1% in the lowest decile (< R$23.8) to ~9.2% in the highest (> R$229.8).
  The continuous `price` feature will be retained as-is, with no binary flag needed
  given the absence of abrupt threshold effects.

- **Freight value**: retained as an independent feature alongside `price`. Although
  correlated with product weight, it may carry unique signal from route distance,
  remote delivery zones, or shipping modalities. Multicollinearity is not a concern
  given the tree-based modeling approach. Feature importance post-training will
  determine whether it is dropped during feature selection.

In [ ]:
# Late rate by seller
seller_late = (
    df[df["is_late"].notna()]
    .groupby("seller_id")
    .agg(
        total_orders=("order_id", "nunique"),
        late_rate=("is_late", "mean"),
    )
    .query("total_orders >= 30")
    .sort_values("late_rate", ascending=False)
    .reset_index()
)

print(seller_late.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top 20 worst sellers
sns.barplot(
    data=seller_late.head(20),
    x="late_rate",
    y="seller_id",
    color="steelblue",
    ax=axes[0],
)
axes[0].set_title("Top 20 sellers – highest late rate")
axes[0].set_xlabel("Late rate")
axes[0].set_ylabel("Seller ID")
axes[0].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

# Distribution of late rate across all sellers
axes[1].hist(seller_late["late_rate"], bins=30, color="steelblue", edgecolor="white")
axes[1].set_title("Late rate distribution – all sellers (≥30 orders)")
axes[1].set_xlabel("Late rate")
axes[1].set_ylabel("Number of sellers")

plt.tight_layout()
plt.show()

### Seller findings

- **Seller late rate**: distribution is right-skewed, with most sellers concentrated
  between 3–10% late rate. A long tail reaches up to ~34%, indicating that a small
  subset of sellers is responsible for a disproportionate share of delays.
  The top 20 worst sellers range from 21% to 34%, well above the global average.
  Given high cardinality, `seller_id` will be encoded via k-fold target encoding,
  expected to be a strong predictive feature.

In [ ]:
review_late = (
    df[df["is_late"].notna()]
    .groupby("review_score")
    .agg(
        total_orders=("order_id", "count"),
        late_rate=("is_late", "mean"),
    )
    .reset_index()
)

print(review_late)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=review_late,
    x="review_score",
    y="late_rate",
    color="steelblue",
    ax=axes[0],
)
axes[0].set_title("Late rate by review score")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(xmax=1))

sns.barplot(
    data=review_late,
    x="review_score",
    y="total_orders",
    color="steelblue",
    ax=axes[1],
)
axes[1].set_title("Order volume by review score")

plt.tight_layout()
plt.show()

### Review score findings

- **Review score**: strong monotonic relationship with late rate. Score 1 reaches
  32% late rate — over 10x higher than score 5 (3%). The relationship is smooth
  and consistent across all levels, confirming `review_score` can be used as a
  continuous numeric feature without encoding. Volume is heavily concentrated
  at score 5 (~63k orders), suggesting most customers are satisfied, while
  dissatisfied reviews are disproportionately driven by late deliveries.